# ADoRE-DoE Phase 4：RSM 建模與診斷

二階模型 → 係數表 / ANOVA / Lack-of-Fit / 模型統計量 → 方程式 → Stepwise → 診斷。
> 響應由**合成函數（Synthetic）**產生；實際使用時請換成真實模擬結果。需要 `pip install -e ".[plot]"`。

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import adore_doe as ad
from adore_doe import design, rsm
os.makedirs('output', exist_ok=True)

## 1. 設計與資料（CCD，3 個中心點，才有純誤差可做 Lack-of-Fit）

In [ ]:
factors = [ad.Factor("Width", 10, 50, unit="nm"), ad.Factor("Spacing", 5, 30, unit="nm"), ad.Factor("Thickness", 20, 80, unit="nm")]
ds = design.ccd(factors, center_points=3, seed=1)
Z = ds.coded
rng = np.random.default_rng(7)
ds.add_response("Capacitance", 2.0 + 0.8*Z[:,0] - 0.5*Z[:,1] + 0.3*Z[:,0]*Z[:,1] - 0.6*Z[:,2]**2 + 0.05*rng.standard_normal(ds.n_runs))
print(ds.n_runs, "runs;", ds.n_replicated_runs, "replicated (centre) runs")

## 2. 完整二階模型：係數、模型統計量

In [ ]:
res = rsm.fit(ds, "Capacitance", model="quadratic")
res.coef_table().round(4)

In [ ]:
res.fit_statistics().round(4)

## 3. ANOVA（Type III）與 Lack-of-Fit

In [ ]:
res.anova().round(4)

只有 1 個中心點的 CCD（ADR-010 預設）沒有純誤差，Lack-of-Fit 會明確回報「不可計算」：

In [ ]:
one = design.ccd(factors, center_points=1, seed=1)
one.add_response("y", 2 + 0.8*one.coded[:,0] - 0.6*one.coded[:,2]**2 + 0.05*np.random.default_rng(1).standard_normal(one.n_runs))
rsm.fit(one, "y").anova().set_index("Source").loc[["Lack of Fit", "Pure Error"], ["df", "note"]]

## 4. 明確方程式（coded 與實際單位）

In [ ]:
print(res.equation(coded=True, digits=4)); print(); print(res.equation(coded=False, digits=4))

## 5. Stepwise（p 值 0.10，strong hierarchy）與 AICc

In [ ]:
sel = {"method": "stepwise", "criterion": "p", "alpha_in": 0.10, "alpha_out": 0.10, "hierarchy": "strong"}
r_p = rsm.fit(ds, "Capacitance", model="quadratic", selection=sel)
r_a = rsm.fit(ds, "Capacitance", model="quadratic", selection={**sel, "criterion": "aicc"})
print("p  :", r_p.term_names); print("AICc:", r_a.term_names)
print([ (h['action'], h['term']) for h in r_p.selection.history], "|", r_p.selection.stop_reason)

## 6. 診斷

In [ ]:
d = res.diagnostics()
print(d.summary()); d.table.round(3).head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
d.plot_normal(ax=ax[0]); d.plot_residual_vs_predicted(ax=ax[1]); d.plot_predicted_vs_actual(ax=ax[2])
plt.tight_layout(); plt.savefig("output/rsm_diagnostics.png", dpi=150); plt.show()

Shapiro–Wilk 套用在學生化殘差上只是近似（模擬誤報率約 10%，名目 5%），請當作警示而非精確檢定。